# Session Embedding Exploration

Exploration des seances a partir du `2025-09-14`, date a partir de laquelle les donnees cardio sont disponibles.

Objectifs:
- construire un embedding tabulaire simple pour chaque seance
- visualiser les seances qui se ressemblent
- comparer les proximit es apprises avec `session_type`
- tester rapidement si une classification automatique semble realiste


In [108]:
from __future__ import annotations

import os
import re
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.neighbors import NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sqlalchemy import create_engine, text

CUTOFF_DATE = pd.Timestamp("2025-09-14")
FEATURE_COLUMNS = [
    "distance_km",
    "duration_min",
    "pace_min_per_km",
    "avg_hr",
    "elevation_m",
    "active_kcal",
    "z1_min",
    "z2_min",
    "z3_min",
    "z4_min",
    "z5_min",
    "low_intensity_pct",
    "high_intensity_pct",
]
DERIVED_INTERVAL_COLUMNS = [
    "interval_block_count",
    "interval_total_reps",
    "interval_total_work_m",
    "interval_avg_rep_distance_m",
    "interval_avg_recovery_m",
    "interval_work_recovery_ratio",
    "interval_has_recovery",
    "interval_has_pace_target",
    "interval_has_intensity_target",
]
ENRICHED_FEATURE_COLUMNS = FEATURE_COLUMNS + DERIVED_INTERVAL_COLUMNS


In [109]:
ROOT = Path.cwd()
if ROOT.name != "HealthCoachBackend":
    ROOT = ROOT / "HealthCoachBackend"

load_dotenv(ROOT / ".env")

DATABASE_URL = os.getenv("DATABASE_URL")
USER_ID = os.getenv("USER_ID") or os.getenv("DEFAULT_USER_ID")

if not DATABASE_URL:
    raise RuntimeError("DATABASE_URL manquant dans HealthCoachBackend/.env")

engine = create_engine(DATABASE_URL, pool_pre_ping=True)

print({"root": str(ROOT), "user_id": USER_ID, "cutoff_date": str(CUTOFF_DATE.date())})


{'root': '/Users/albanecoiffe/Library/Mobile Documents/com~apple~CloudDocs/Documents/Projet/HealthCoach/HealthCoachBackend/exploration/HealthCoachBackend', 'user_id': 'f90a87bf-2104-4456-8a54-b42c307337e7', 'cutoff_date': '2025-09-14'}


In [110]:
with engine.connect() as conn:
    available_columns = {
        row[0]
        for row in conn.execute(
            text(
                """
                SELECT column_name
                FROM information_schema.columns
                WHERE table_schema = 'public' AND table_name = 'run_sessions'
                """
            )
        )
    }

    select_columns = [
        "id",
        "user_id",
        "start_time",
        "distance_km",
        "duration_min",
        "avg_hr",
        "elevation_m",
        "active_kcal",
        "z1_min",
        "z2_min",
        "z3_min",
        "z4_min",
        "z5_min",
    ]

    optional_columns = ["session_type", "session_detail"]
    for column in optional_columns:
        if column in available_columns:
            select_columns.append(column)

    where_clauses = ["start_time >= :cutoff_date"]
    params = {"cutoff_date": CUTOFF_DATE.to_pydatetime()}
    if USER_ID:
        where_clauses.append("user_id = :user_id")
        params["user_id"] = USER_ID

    query = f"""
        SELECT {', '.join(select_columns)}
        FROM run_sessions
        WHERE {' AND '.join(where_clauses)}
        ORDER BY start_time ASC
    """

    sessions = pd.read_sql(text(query), conn, params=params)

for column in optional_columns:
    if column not in sessions.columns:
        sessions[column] = pd.Series(dtype="object")

sessions["start_time"] = pd.to_datetime(sessions["start_time"], utc=True, errors="coerce")
sessions = sessions.dropna(subset=["start_time", "distance_km", "duration_min"]).copy()
sessions = sessions[sessions["duration_min"] > 0].copy()
sessions["distance_km"] = sessions["distance_km"].astype(float)
sessions["duration_min"] = sessions["duration_min"].astype(float)
sessions["pace_min_per_km"] = sessions["duration_min"] / sessions["distance_km"].replace(0, np.nan)
sessions["low_intensity_pct"] = (
    sessions[["z1_min", "z2_min", "z3_min"]].fillna(0).sum(axis=1) / sessions["duration_min"].replace(0, np.nan)
)
sessions["high_intensity_pct"] = (
    sessions[["z4_min", "z5_min"]].fillna(0).sum(axis=1) / sessions["duration_min"].replace(0, np.nan)
)
sessions["session_type"] = sessions["session_type"].fillna("unlabeled")
sessions["session_detail"] = sessions["session_detail"].fillna("")
sessions["date_local"] = sessions["start_time"].dt.tz_convert("Europe/Paris")
sessions["session_label"] = sessions["date_local"].dt.strftime("%Y-%m-%d") + " | " + sessions["session_type"]

print(f"{len(sessions)} seances chargees")
sessions.head()


100 seances chargees


,id,user_id,start_time,distance_km,duration_min,avg_hr,elevation_m,active_kcal,z1_min,z2_min,z3_min,z4_min,z5_min,session_type,session_detail,pace_min_per_km,low_intensity_pct,high_intensity_pct,date_local,session_label
0,e6444938-61cf-4f6b-864e-077468839abf,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 07:56:56+00:00,1.006600,6.514466,135.106013,2.56,42.264026,4.183333,3.316667,0.000000,0.000000,0.0,footing,,6.471751,1.151284,0.000000,2025-09-14 09:56:56+02:00,2025-09-14 | footing
1,0c4bbeb7-073c-487e-b49f-d54de45e1ceb,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 08:26:12+00:00,5.052251,32.672392,160.876990,8.51,223.886737,0.000000,21.250000,12.195833,0.000000,0.0,footing,,6.466898,1.023673,0.000000,2025-09-14 10:26:12+02:00,2025-09-14 | footing
2,aea50235-c0f1-4ed5-99b9-bc12f19fe212,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 08:59:44+00:00,3.013890,15.612216,176.997594,4.65,132.558014,0.000000,0.000000,5.983508,6.970833,0.0,fractionné,1*3000 R000 5:11/km,5.180089,0.383258,0.446499,2025-09-14 10:59:44+02:00,2025-09-14 | fractionné
3,14de452b-ccdf-45ab-8c8b-713cd82ae374,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-14 09:16:13+00:00,0.821181,5.411381,162.782103,2.14,34.139432,0.000000,0.000000,2.675000,0.000000,0.0,footing,,6.589755,0.494329,0.000000,2025-09-14 11:16:13+02:00,2025-09-14 | footing
4,b7a921ad-4ce1-42ce-9324-9eeea4f75e49,f90a87bf-2104-4456-8a54-b42c307337e7,2025-09-16 17:34:22+00:00,12.835037,77.284413,159.908975,70.43,586.003636,0.000000,7.280263,37.969466,20.850848,0.0,fractionné,2*4000 R500 5:12/km,6.021363,0.585496,0.269794,2025-09-16 19:34:22+02:00,2025-09-16 | fractionné


In [111]:
summary = pd.DataFrame(
    {
        "n_sessions": [len(sessions)],
        "n_labeled": [(sessions["session_type"] != "unlabeled").sum()],
        "n_session_types": [sessions.loc[sessions["session_type"] != "unlabeled", "session_type"].nunique()],
        "date_min": [sessions["date_local"].min()],
        "date_max": [sessions["date_local"].max()],
    }
)
display(summary)
display(sessions["session_type"].value_counts(dropna=False).rename("count").to_frame())


,n_sessions,n_labeled,n_session_types,date_min,date_max
0,100,100,5,2025-09-14 09:56:56+02:00,2026-04-26 09:00:36+02:00


,count
session_type,
footing,51
fractionné,26
sortie longue,20
semi marathon,2
marathon,1


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur les donnees

- Le jeu contient `100` seances entre le 14 septembre 2025 et le 26 avril 2026. Pour une premiere etude individuelle, c est deja une base interessante.
- Les labels sont exploitables mais desequilibres: `footing` (`51`), `fractionné` (`26`), `sortie longue` (`20`), puis seulement `semi marathon` (`2`) et `marathon` (`1`). Pour l apprentissage supervise, les trois premieres classes sont les seules vraiment evaluables.
- On voit plusieurs enregistrements le meme jour au debut de la periode. Cela peut correspondre a des blocs separes de la meme seance. Si vous voulez ensuite raisonner au niveau "vraie seance", il faudra peut-etre tester un regroupement par jour ou par proximite temporelle.
- Les ratios d intensite peuvent depasser `1` sur certaines lignes, par exemple `low_intensity_pct`. Ce n empeche pas l exploration de marcher, mais cela indique un petit sujet de qualite de donnees a corriger avant de figer un pipeline de production.


## Embedding tabulaire

Ici, l'embedding de base est simplement le vecteur de features numeriques normalisees. C'est suffisant pour explorer les ressemblances entre seances avant de passer a quelque chose de plus sophistique.

In [112]:
embedding_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

X = sessions[FEATURE_COLUMNS].copy()
embedding = embedding_pipeline.fit_transform(X)

embedding_df = pd.DataFrame(embedding, columns=FEATURE_COLUMNS, index=sessions.index)
embedding_df.head()


,distance_km,duration_min,pace_min_per_km,avg_hr,elevation_m,active_kcal,z1_min,z2_min,z3_min,z4_min,z5_min,low_intensity_pct,high_intensity_pct
0,-1.333936,-1.451138,0.069766,-1.518097,-1.220246,-1.350563,-0.306062,-1.054464,-0.769111,-0.403702,-0.257041,1.464558,-0.599799
1,-0.692349,-0.722601,0.060936,0.782210,-1.011141,-0.717776,-0.861190,-0.011726,-0.323890,-0.403702,-0.257041,0.779540,-0.599799
2,-1.015607,-1.197752,-2.280869,2.221128,-1.146796,-1.035973,-0.861190,-1.247312,-0.550677,-0.080878,-0.257041,-2.658207,2.014064
3,-1.363342,-1.481860,0.284518,0.952259,-1.235006,-1.378870,-0.861190,-1.247312,-0.671457,-0.403702,-0.257041,-2.061981,-0.599799
4,0.541898,0.519909,-0.749874,0.695805,1.164951,0.543866,-0.861190,-0.823999,0.617001,0.561915,-0.257041,-1.572593,0.979610


In [113]:
pca = PCA(n_components=2, random_state=42)
coords = pca.fit_transform(embedding)

viz_df = sessions[["session_label", "session_type", "session_detail", "date_local"]].copy()
viz_df["pc1"] = coords[:, 0]
viz_df["pc2"] = coords[:, 1]
viz_df["distance_km"] = sessions["distance_km"].round(2)
viz_df["duration_min"] = sessions["duration_min"].round(1)
viz_df["avg_hr"] = sessions["avg_hr"].round(0)
viz_df["high_intensity_pct"] = sessions["high_intensity_pct"].round(3)

fig = px.scatter(
    viz_df,
    x="pc1",
    y="pc2",
    color="session_type",
    hover_data=["date_local", "distance_km", "duration_min", "avg_hr", "high_intensity_pct", "session_detail"],
    title="Projection PCA des embeddings de seance",
)
fig.update_traces(marker={"size": 10, "opacity": 0.8})
fig.show()

print("Variance expliquee:", pca.explained_variance_ratio_.round(3))


Variance expliquee: [0.475 0.159]


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur la projection PCA

- Les deux premieres composantes expliquent environ `63.4%` de la variance (`47.5%` + `15.9%`). Pour une visualisation 2D, c est correct: on garde une part importante de la structure des donnees.
- Si les couleurs montrent deja des groupes assez distincts dans le nuage, c est un bon signe que les features cardio + allure + volume portent deja une information de type de seance.
- Il faut cependant rester prudent: une belle separation visuelle en PCA ne garantit pas qu un classifieur generalisera parfaitement. La vraie mesure reste celle de la validation croisee plus bas.


In [114]:
neighbors = NearestNeighbors(n_neighbors=min(6, len(sessions)), metric="euclidean")
neighbors.fit(embedding)
distances, indices = neighbors.kneighbors(embedding)

def similar_sessions(session_index: int, top_k: int = 5) -> pd.DataFrame:
    rows = []
    for rank, (distance, idx) in enumerate(zip(distances[session_index][1 : top_k + 1], indices[session_index][1 : top_k + 1]), start=1):
        row = sessions.iloc[idx]
        rows.append(
            {
                "rank": rank,
                "distance_in_embedding_space": round(float(distance), 3),
                "start_time": row["date_local"],
                "session_type": row["session_type"],
                "session_detail": row["session_detail"],
                "distance_km": row["distance_km"],
                "duration_min": row["duration_min"],
                "avg_hr": row["avg_hr"],
                "high_intensity_pct": row["high_intensity_pct"],
            }
        )
    return pd.DataFrame(rows)

reference_index = len(sessions) - 1
display(sessions.iloc[[reference_index]][["date_local", "session_type", "session_detail", "distance_km", "duration_min", "avg_hr", "high_intensity_pct"]])
similar_sessions(reference_index)


,date_local,session_type,session_detail,distance_km,duration_min,avg_hr,high_intensity_pct
99,2026-04-26 09:00:36+02:00,footing,,4.016815,26.355755,144.104608,0.0


,rank,distance_in_embedding_space,start_time,session_type,session_detail,distance_km,duration_min,avg_hr,high_intensity_pct
0,1,1.228,2026-02-02 18:15:22+01:00,footing,,4.570502,31.634559,143.895680,0.000000
1,2,1.412,2025-12-01 09:01:49+01:00,fractionné,6*400 R100 4:20/km,7.042671,46.399299,139.370393,0.000431
2,3,1.461,2026-03-17 12:07:44+01:00,footing,,4.321521,29.201466,151.236814,0.000000
3,4,1.477,2025-11-25 17:10:52+01:00,footing,,5.032676,35.028532,145.679298,0.000000
4,5,1.548,2026-03-24 18:49:29+01:00,footing,,4.190071,29.497249,143.329558,0.000000


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur les voisins proches

- Pour la seance de reference du `2026-04-26`, les voisins les plus proches sont majoritairement des `footing`, ce qui est coherent avec le label manuel.
- Un `fractionné` apparait quand meme parmi les plus proches, mais avec une intensite quasi nulle (`high_intensity_pct` proche de `0`). Cela suggere qu une partie de vos fractionnes les plus faciles ressemblent deja a des footings courts dans l espace de features actuel.
- C est exactement le type de cas ou le `session_detail` pourra apporter de la valeur: deux seances peuvent se ressembler physiologiquement tout en etant structurellement differentes.


In [115]:
n_clusters = min(5, max(2, sessions.loc[sessions["session_type"] != "unlabeled", "session_type"].nunique() or 3))
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=20)
sessions["cluster"] = kmeans.fit_predict(embedding)

cluster_profile = (
    sessions.groupby("cluster")[FEATURE_COLUMNS]
    .mean()
    .round(2)
)
display(cluster_profile)

labeled = sessions[sessions["session_type"] != "unlabeled"].copy()
if not labeled.empty:
    display(pd.crosstab(labeled["cluster"], labeled["session_type"], normalize="index").round(2))
else:
    print("Aucune seance labelisee pour comparer clusters et session_type.")


,distance_km,duration_min,pace_min_per_km,avg_hr,elevation_m,active_kcal,z1_min,z2_min,z3_min,z4_min,z5_min,low_intensity_pct,high_intensity_pct
cluster,,,,,,,,,,,,,
0,19.18,114.44,5.96,164.75,62.08,876.69,2.90,12.56,87.54,13.94,0.00,0.91,0.12
1,4.50,31.04,6.91,145.10,16.51,205.05,9.86,15.56,4.96,0.35,0.00,0.97,0.01
2,28.45,158.44,5.51,152.01,89.30,1279.93,5.07,0.81,41.05,110.44,1.29,0.29,0.71
3,10.02,60.23,6.02,162.39,26.05,459.66,4.10,14.88,21.04,18.35,0.96,0.66,0.32
4,10.79,67.70,6.28,151.52,59.00,492.82,4.64,37.98,20.63,2.35,0.00,0.94,0.03


session_type,footing,fractionné,marathon,semi marathon,sortie longue
cluster,,,,,
0,0.0,0.00,0.00,0.00,1.00
1,0.9,0.10,0.00,0.00,0.00
2,0.0,0.00,0.33,0.67,0.00
3,0.0,0.94,0.00,0.00,0.06
4,0.5,0.17,0.00,0.00,0.33


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur les clusters

- Le clustering retrouve deja des groupes tres interpretable: un cluster quasi pur `sortie longue`, un cluster tres majoritairement `footing`, et un cluster tres majoritairement `fractionné`.
- Le cluster `3` est presque un cluster `fractionné` pur (`94%`), ce qui est un excellent signal: meme sans donner le label au modele, certaines seances intenses se regroupent naturellement.
- Le cluster `4` est plus melange. Il ressemble a une zone de transition entre footing soutenu, sortie longue moderee et seances plus structurees mais pas tres dures. Ce cluster est probablement le plus interessant a etudier qualitativement.
- Le cluster `2` capture surtout vos efforts tres longs / tres denses (`semi marathon`, `marathon`). Comme ces labels sont tres rares, il vaut mieux les traiter pour l instant comme des cas speciaux plutot que comme des classes de classification standard.


## Classification supervisee

Le but ici n'est pas de produire un modele final, mais de voir si `session_type` est suffisamment coherent pour etre predit a partir des variables de seance.

In [116]:
labeled = sessions[sessions["session_type"] != "unlabeled"].copy()
label_counts = labeled["session_type"].value_counts()
eligible_labels = label_counts[label_counts >= 3].index
train_df = labeled[labeled["session_type"].isin(eligible_labels)].copy()

print("Labels retenus:")
display(label_counts.to_frame("count"))

if train_df["session_type"].nunique() < 2:
    print("Pas assez de types de seance representes pour entrainer une classification.")
else:
    X_train = train_df[FEATURE_COLUMNS]
    y_train = train_df["session_type"]

    cv_splits = min(5, int(y_train.value_counts().min()))
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    classifier = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, multi_class="auto")),
        ]
    )

    f1_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="f1_macro")
    bal_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="balanced_accuracy")
    y_pred = cross_val_predict(classifier, X_train, y_train, cv=cv)

    print(f"Macro F1 mean: {f1_scores.mean():.3f} +/- {f1_scores.std():.3f}")
    print(f"Balanced accuracy mean: {bal_scores.mean():.3f} +/- {bal_scores.std():.3f}")
    print()
    print(classification_report(y_train, y_pred, digits=3))

    cm = pd.DataFrame(
        confusion_matrix(y_train, y_pred, labels=sorted(y_train.unique())),
        index=[f"true:{label}" for label in sorted(y_train.unique())],
        columns=[f"pred:{label}" for label in sorted(y_train.unique())],
    )
    display(cm)


Labels retenus:


,count
session_type,
footing,51
fractionné,26
sortie longue,20
semi marathon,2
marathon,1


Macro F1 mean: 0.862 +/- 0.100
Balanced accuracy mean: 0.854 +/- 0.112

               precision    recall  f1-score   support

      footing      0.891     0.961     0.925        51
   fractionné      0.840     0.808     0.824        26
sortie longue      0.941     0.800     0.865        20

     accuracy                          0.887        97
    macro avg      0.891     0.856     0.871        97
 weighted avg      0.888     0.887     0.885        97



,pred:footing,pred:fractionné,pred:sortie longue
true:footing,49,2,0
true:fractionné,4,21,1
true:sortie longue,2,2,16


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur la classification

- Le resultat est deja fort pour une baseline simple: `Macro F1 ~ 0.86` et `balanced accuracy ~ 0.85`. Cela veut dire qu une classification automatique de type de seance est realiste avec vos seules features numeriques.
- `footing` est la classe la plus facile a reconnaitre (`f1 = 0.925`). C est coherent: ces seances sont nombreuses et assez homogenes.
- `fractionné` et `sortie longue` sont plus souvent confondus entre eux. Cela peut venir de deux phenomenes: des seances longues avec blocs d allure, et des fractionnes longs de type seuil / allure marathon.
- Le prochain gain probable ne viendra pas d un modele plus complexe tout de suite, mais de meilleures features: parsing de `session_detail`, prise en compte du nombre de repetitions, de la distance des blocs, et eventuellement regroupement echauffement / bloc / retour au calme.
- Tant que `marathon` et `semi marathon` restent quasi absents, il ne faut pas essayer de les predire comme classes autonomes. Mieux vaut soit les exclure du training, soit les rabattre temporairement sur une categorie effort long / course objectif.


In [117]:
def parse_interval_blocks(detail: str) -> list[dict]:
    detail = (detail or "").strip()
    if not detail:
        return []

    blocks = []
    for raw_block in [part.strip() for part in detail.split(",") if part.strip()]:
        match = re.match(
            r"(?P<reps>\d+)\s*[x\*]\s*(?P<distance>\d+)\s*m?\s*(?:R\s*(?P<recovery>\d+))?\s*(?P<target>.*)",
            raw_block,
            flags=re.IGNORECASE,
        )
        if not match:
            blocks.append({"raw_block": raw_block})
            continue

        blocks.append(
            {
                "raw_block": raw_block,
                "reps": int(match.group("reps")),
                "work_distance_m": int(match.group("distance")),
                "recovery_m": int(match.group("recovery")) if match.group("recovery") else np.nan,
                "target": match.group("target").strip() or None,
            }
        )

    return blocks

def summarize_interval_detail(detail: str) -> dict:
    blocks = parse_interval_blocks(detail)
    parsed_blocks = [b for b in blocks if "reps" in b and "work_distance_m" in b]

    if not parsed_blocks:
        return {col: 0.0 for col in DERIVED_INTERVAL_COLUMNS}

    total_reps = float(sum(b["reps"] for b in parsed_blocks))
    total_work_m = float(sum(b["reps"] * b["work_distance_m"] for b in parsed_blocks))
    recovery_values = [float(b["recovery_m"]) for b in parsed_blocks if pd.notna(b.get("recovery_m"))]
    pace_target_regex = re.compile(r"\d{1,2}:\d{2}\s*/\s*km|\d{1,2}min/km", flags=re.IGNORECASE)
    intensity_tokens = ("am", "as10", "as21", "as42", "seuil", "tempo", "vma")
    targets = [str(b.get("target") or "").strip().lower() for b in parsed_blocks]

    avg_rep_distance_m = total_work_m / total_reps if total_reps > 0 else 0.0
    avg_recovery_m = float(np.mean(recovery_values)) if recovery_values else 0.0
    work_recovery_ratio = avg_rep_distance_m / avg_recovery_m if avg_recovery_m > 0 else 0.0

    return {
        "interval_block_count": float(len(parsed_blocks)),
        "interval_total_reps": total_reps,
        "interval_total_work_m": total_work_m,
        "interval_avg_rep_distance_m": avg_rep_distance_m,
        "interval_avg_recovery_m": avg_recovery_m,
        "interval_work_recovery_ratio": work_recovery_ratio,
        "interval_has_recovery": float(bool(recovery_values)),
        "interval_has_pace_target": float(any(pace_target_regex.search(t) for t in targets)),
        "interval_has_intensity_target": float(any(any(token in t for token in intensity_tokens) for t in targets)),
    }

interval_rows = []
derived_feature_rows = []
for idx, row in sessions.iterrows():
    detail = row["session_detail"]
    derived = summarize_interval_detail(detail)
    derived_feature_rows.append({"session_index": idx, **derived})

    if str(detail).strip():
        for block in parse_interval_blocks(detail):
            block["date_local"] = row["date_local"]
            block["session_type"] = row["session_type"]
            block["session_detail"] = row["session_detail"]
            interval_rows.append(block)

derived_interval_df = pd.DataFrame(derived_feature_rows).set_index("session_index")
sessions = sessions.join(derived_interval_df)
parsed_intervals = pd.DataFrame(interval_rows)
display(sessions[["session_type", "session_detail"] + DERIVED_INTERVAL_COLUMNS].head(12))
parsed_intervals.head(20)


,session_type,session_detail,interval_block_count,interval_total_reps,interval_total_work_m,interval_avg_rep_distance_m,interval_avg_recovery_m,interval_work_recovery_ratio,interval_has_recovery,interval_has_pace_target,interval_has_intensity_target
0,footing,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,footing,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,fractionné,1*3000 R000 5:11/km,1.0,1.0,3000.0,3000.0,0.0,0.0,1.0,1.0,0.0
3,footing,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,fractionné,2*4000 R500 5:12/km,1.0,2.0,8000.0,4000.0,500.0,8.0,1.0,1.0,0.0
5,sortie longue,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
6,fractionné,3*3000 R500 5:08/km,1.0,3.0,9000.0,3000.0,500.0,6.0,1.0,1.0,0.0
7,footing,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
8,sortie longue,,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
9,fractionné,1*4000 R000 5:08/km,1.0,1.0,4000.0,4000.0,0.0,0.0,1.0,1.0,0.0


,raw_block,reps,work_distance_m,recovery_m,target,date_local,session_type,session_detail
0,1*3000 R000 5:11/km,1.0,3000.0,0.0,5:11/km,2025-09-14 10:59:44+02:00,fractionné,1*3000 R000 5:11/km
1,2*4000 R500 5:12/km,2.0,4000.0,500.0,5:12/km,2025-09-16 19:34:22+02:00,fractionné,2*4000 R500 5:12/km
2,3*3000 R500 5:08/km,3.0,3000.0,500.0,5:08/km,2025-09-23 19:08:18+02:00,fractionné,3*3000 R500 5:08/km
3,1*4000 R000 5:08/km,1.0,4000.0,0.0,5:08/km,2025-09-29 18:57:10+02:00,fractionné,1*4000 R000 5:08/km
4,2*4000 R500 5:08/km,2.0,4000.0,500.0,5:08/km,2025-10-06 07:47:45+02:00,fractionné,2*4000 R500 5:08/km
5,1*3000 R000 5:09/km,1.0,3000.0,0.0,5:09/km,2025-10-11 10:38:27+02:00,sortie longue,1*3000 R000 5:09/km
6,2*5000 R500 5:15/km,2.0,5000.0,500.0,5:15/km,2025-10-13 07:50:51+02:00,fractionné,2*5000 R500 5:15/km
7,2*2000 R300 5:09/km,2.0,2000.0,300.0,5:09/km,2025-10-20 18:19:10+02:00,fractionné,2*2000 R300 5:09/km
8,3*2000 R300 5:09/km,3.0,2000.0,300.0,5:09/km,2025-10-28 07:51:41+01:00,fractionné,3*2000 R300 5:09/km
9,1*3000 R000 5:13/km,1.0,3000.0,0.0,5:13/km,2025-11-02 08:54:24+01:00,sortie longue,1*3000 R000 5:13/km


<!-- AUTO-ANALYSIS-COMMENT -->
## Commentaire sur le parsing de `session_detail`

- Le parsing est maintenant correct sur les formats observes, y compris `4*800m R200 4:50/km`. Les repetitions, distances de bloc, recuperations et cibles d allure sont bien extraites.
- Les nouvelles features derivees sont pertinentes: nombre de blocs, repetitions totales, volume de travail, distance moyenne de repetition, recuperation moyenne, ratio travail / recuperation, presence de recup, presence d une cible allure, presence d une cible d intensite.
- Ces features ne sont non nulles que sur environ `26%` a `30%` des seances, ce qui est logique: elles portent surtout de l information sur les seances structurees, donc principalement les `fractionné`.
- Le fait de voir quelques `sortie longue` avec un detail de type `1*3000 ...` est tres instructif: vos labels manuels capturent bien l intention globale de la seance, alors que `session_detail` capture sa structure interne. Les deux niveaux d information sont complementaires.


## Re-test avec features derivees de `session_detail`

On compare ici la baseline numerique initiale avec une version enrichie par la structure des fractionnes.


In [118]:
labeled = sessions[sessions["session_type"] != "unlabeled"].copy()
label_counts = labeled["session_type"].value_counts()
eligible_labels = label_counts[label_counts >= 3].index
train_df = labeled[labeled["session_type"].isin(eligible_labels)].copy()

def evaluate_feature_set(feature_columns: list[str], name: str) -> dict:
    X_train = train_df[feature_columns]
    y_train = train_df["session_type"]

    cv_splits = min(5, int(y_train.value_counts().min()))
    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=42)

    classifier = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(max_iter=2000, multi_class="auto")),
        ]
    )

    f1_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="f1_macro")
    bal_scores = cross_val_score(classifier, X_train, y_train, cv=cv, scoring="balanced_accuracy")
    y_pred = cross_val_predict(classifier, X_train, y_train, cv=cv)

    return {
        "name": name,
        "feature_count": len(feature_columns),
        "f1_mean": float(f1_scores.mean()),
        "f1_std": float(f1_scores.std()),
        "balanced_accuracy_mean": float(bal_scores.mean()),
        "balanced_accuracy_std": float(bal_scores.std()),
        "y_pred": y_pred,
        "features": feature_columns,
    }

if train_df["session_type"].nunique() < 2:
    print("Pas assez de types de seance representes pour comparer les feature sets.")
else:
    baseline_result = evaluate_feature_set(FEATURE_COLUMNS, "baseline")
    enriched_result = evaluate_feature_set(ENRICHED_FEATURE_COLUMNS, "baseline + interval details")

    comparison = pd.DataFrame([baseline_result, enriched_result]).drop(columns=["y_pred", "features"])
    for col in ["f1_mean", "f1_std", "balanced_accuracy_mean", "balanced_accuracy_std"]:
        comparison[col] = comparison[col].round(3)
    display(comparison)

    print("Classification report - enriched model")
    print(classification_report(train_df["session_type"], enriched_result["y_pred"], digits=3))

    cm = pd.DataFrame(
        confusion_matrix(train_df["session_type"], enriched_result["y_pred"], labels=sorted(train_df["session_type"].unique())),
        index=[f"true:{label}" for label in sorted(train_df["session_type"].unique())],
        columns=[f"pred:{label}" for label in sorted(train_df["session_type"].unique())],
    )
    display(cm)

    importance_proxy = pd.DataFrame(
        {
            "feature": ENRICHED_FEATURE_COLUMNS,
            "missing_rate": train_df[ENRICHED_FEATURE_COLUMNS].isna().mean().values,
            "non_zero_rate": (train_df[ENRICHED_FEATURE_COLUMNS].fillna(0) != 0).mean().values,
        }
    ).sort_values(["non_zero_rate", "missing_rate"], ascending=[False, True])
    display(importance_proxy)


,name,feature_count,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,baseline,13,0.862,0.100,0.854,0.112
1,baseline + interval details,22,0.949,0.049,0.937,0.057


Classification report - enriched model
               precision    recall  f1-score   support

      footing      0.944     1.000     0.971        51
   fractionné      0.962     0.962     0.962        26
sortie longue      1.000     0.850     0.919        20

     accuracy                          0.959        97
    macro avg      0.969     0.937     0.951        97
 weighted avg      0.960     0.959     0.958        97



,pred:footing,pred:fractionné,pred:sortie longue
true:footing,51,0,0
true:fractionné,1,25,0
true:sortie longue,2,1,17


,feature,missing_rate,non_zero_rate
0,distance_km,0.000000,1.000000
1,duration_min,0.000000,1.000000
2,pace_min_per_km,0.000000,1.000000
3,avg_hr,0.000000,1.000000
5,active_kcal,0.000000,1.000000
11,low_intensity_pct,0.000000,1.000000
7,z2_min,0.000000,0.958763
4,elevation_m,0.051546,0.948454
6,z1_min,0.000000,0.886598
8,z3_min,0.000000,0.752577


## Commentaire sur le re-test enrichi

- Le gain est net: on passe de `Macro F1 = 0.862` a `0.949`, et de `balanced accuracy = 0.854` a `0.937`. Ce n est pas un petit bruit statistique: `session_detail` apporte une vraie information predictive.
- `footing` devient parfaitement reconnu dans cette validation croisee (`51/51` bien classes). `fractionné` monte aussi tres haut (`25/26` bien classes).
- La confusion `fractionné` / `sortie longue` baisse fortement par rapport a la baseline. C etait exactement le signal a regarder, et il confirme que la structure des blocs aide a separer les seances physiologiquement proches.
- Les erreurs residuelles sont surtout cote `sortie longue`, avec encore quelques seances re-classees en `footing` ou `fractionné`. Cela suggere que certaines sorties longues ont un profil cardio proche d autres categories, ou qu elles contiennent des blocs rapides qui brouillent la frontiere.
- Conclusion pratique: une classification automatique de seance est deja realiste chez vous, et elle devient tres solide des qu on ajoute des features structurelles issues de `session_detail`. La prochaine etape utile serait de stabiliser le nettoyage des ratios et de tester un regroupement des blocs appartenant possiblement a une meme seance.
